# 11. Gamma-Risk Framework

Production-grade gamma-risk metrics for a Uni v3 LP position. Moves from FGY (mean-based) to GSR (risk-based).

```
PnL = Fees − IL_realized
IL_realized = Σ ½ · |Γ_t| · P_t² · r_t²
```

**Pipeline** (engine: `engines.gamma_risk_engine`)
1. Γ_t time series (signed + abs)
2. realized IL increments (HF) + daily aggregate
3. RV, BV, Jump daily
4. IL decomposition: diffusion vs jump (proportional to BV/RV)
5. Core metrics: FGY, GSR, σ*, GammaPnL series
6. Tail diagnostics: % profitable, worst-5 contribution, skew
7. Bonus plots: IL distribution, cumulative GammaPnL, jump-day highlights

**Pool note.** The user prompt referenced ETH/USDC **0.3%**; our indexed pool is USDC/WETH **0.05%**. The framework is identical.

**Position note.** No real per-position L/Pa/Pb feed — we use a synthetic LP centered ±5% on the entry-day price, with `L` set equal to the pool's average active liquidity over the window for concrete numbers.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.db import ClickHouseDB, ClickHouseSettings
from engines.gamma_risk_engine import (
    LpPosition,
    compute_gamma,
    compute_realized_il,
    aggregate_il_daily,
    compute_rv_bv_jumps,
    decompose_il,
    compute_core_metrics,
    compute_tail_diagnostics,
    run_gamma_risk_pipeline,
)

db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL       = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"  # USDC/WETH 0.05%
START_DATE = "2025-10-08"   # extended to 180 days back from END_DATE
END_DATE   = "2026-04-06"
print(f"Pool:   {POOL}")
print(f"Window: {START_DATE} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:   0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
Window: 2025-10-08 → 2026-04-06


<function __main__.<lambda>()>

## §1 — Load inputs (hourly prices/returns, daily fees, daily TVL_active)

In [2]:
# Hourly OHLC + log_return_hourly
intraday_pl = db.read_query(f"""
SELECT hour, close_price, log_return_hourly
FROM mart_price_intraday
WHERE lower(pool) = lower('{POOL}')
  AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
ORDER BY hour
""")
hf = intraday_pl.to_pandas()
hf["hour"] = pd.to_datetime(hf["hour"])
hf = hf.set_index("hour").sort_index()
hf = hf.rename(columns={"close_price": "price", "log_return_hourly": "r"})

# Daily fees (in token0 = USDC)
fees_pl = db.read_query(f"""
SELECT date, fees_token0_equiv AS fees_daily
FROM mart_volume_daily
WHERE lower(pool) = lower('{POOL}')
  AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
ORDER BY date
""")
fees_df = fees_pl.to_pandas()
fees_df["date"] = pd.to_datetime(fees_df["date"])
fees = fees_df.set_index("date")["fees_daily"].astype(float)

# Daily active TVL (in token0 = USDC). For σ* we use the average over the window.
atv_pl = db.read_query(f"""
SELECT date, active_tvl_token0 AS tvl_active
FROM int_active_tvl_daily
WHERE lower(pool) = lower('{POOL}')
  AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
ORDER BY date
""")
atv_df = atv_pl.to_pandas()
atv_df["date"] = pd.to_datetime(atv_df["date"])
tvl_active_series = atv_df.set_index("date")["tvl_active"].astype(float)
tvl_active_avg = float(tvl_active_series.mean())

print(f"Hourly observations: {len(hf):,}")
print(f"Daily fees:          {len(fees)} days")
print(f"Avg active TVL:      ${tvl_active_avg:,.0f}")
hf.head(3)


Hourly observations: 4,332
Daily fees:          181 days
Avg active TVL:      $33,429,511


,price,r
hour,,
2025-10-08 00:00:00,4476.842157,NaN
2025-10-08 01:00:00,4488.632404,0.002630
2025-10-08 02:00:00,4484.359154,-0.000952


## §2 — Define LP position (synthetic, wide range covering the window)


In [3]:
# Token decimals to scale L_raw → L_human
pool_meta = db.read_query(f"""
SELECT token0_decimals, token1_decimals
FROM stg_pools
WHERE lower(pool) = lower('{POOL}')
""").to_pandas().iloc[0]
d0, d1 = int(pool_meta["token0_decimals"]), int(pool_meta["token1_decimals"])

# Pool average active L_raw over the window
active_l_pl = db.read_query(f"""
SELECT avg(active_liquidity) AS l_raw_avg
FROM int_active_liquidity_daily
WHERE lower(pool) = lower('{POOL}')
  AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
""").to_pandas()
L_raw_avg = float(active_l_pl["l_raw_avg"].iloc[0])
L_human = L_raw_avg / 10 ** ((d0 + d1) / 2.0)

# Synthetic LP setup: treat the pool's average active liquidity as a single
# representative LP with a range wide enough to stay in-range across the window.
# Justification: with a 33% price drop on ETH/USDC over Q1 2026, a narrow range
# (e.g. ±5%) goes OOR almost immediately and IL collapses to 0 — masking the
# framework. A wide range exercises the framework end-to-end.
P_min = float(hf["price"].min())
P_max = float(hf["price"].max())
Pa = P_min * 0.80   # 20% margin below realized minimum
Pb = P_max * 1.20   # 20% margin above realized maximum
P_entry = float(hf["price"].iloc[0])

position = LpPosition(L=L_human, Pa=Pa, Pb=Pb)

# Implied "full-range" value at the average price
P_avg = float(hf["price"].mean())
import math
V_FR_avg = 2.0 * L_human * math.sqrt(P_avg)

print(f"Token decimals:    d0={d0} (USDC), d1={d1} (WETH)")
print(f"Pool avg L_raw:    {L_raw_avg:,.4e}")
print(f"L_human:           {L_human:,.2f}")
print()
print(f"Window price range: ${P_min:,.0f} → ${P_max:,.0f}  (entry ${P_entry:,.0f}, avg ${P_avg:,.0f})")
print(f"Position range:    [Pa=${Pa:,.0f}, Pb=${Pb:,.0f}]  (covers full window with 20% margin)")
print(f"V_FR at avg P:     ${V_FR_avg:,.0f}")
print(f"TVL_active avg:    ${tvl_active_avg:,.0f}  (concentration leverage = {V_FR_avg/tvl_active_avg:.1f}×)")


Token decimals:    d0=6 (USDC), d1=18 (WETH)
Pool avg L_raw:    5.4779e+18
L_human:           5,477,872.75

Window price range: $1,816 → $4,541  (entry $4,477, avg $2,851)
Position range:    [Pa=$1,453, Pb=$5,449]  (covers full window with 20% margin)
V_FR at avg P:     $585,001,209
TVL_active avg:    $33,429,511  (concentration leverage = 17.5×)


## §3 — Run pipeline end-to-end

In [4]:
out = run_gamma_risk_pipeline(
    prices=hf["price"],
    returns=hf["r"],
    fees_daily=fees,
    position=position,
    tvl_active_avg=tvl_active_avg,
    normalize_bv=True,   # BNS π/2 normalization → BV/RV → 1 under no jumps
)
hf_df    = out.hf_df
daily_df = out.daily_df
summary  = out.summary
print(f"hf_df:    {hf_df.shape}")
print(f"daily_df: {daily_df.shape}")


hf_df:    (4332, 6)
daily_df: (181, 10)


## §4 — Intermediate outputs

**Gamma series (last 5 hours)**

In [5]:
hf_df.tail(5)[["price", "in_range", "gamma", "abs_gamma", "r", "il_increment"]]


,price,in_range,gamma,abs_gamma,r,il_increment
hour,,,,,,
2026-04-06 07:00:00,2135.307015,1,-27.758210,27.758210,0.003624,831.012816
2026-04-06 08:00:00,2138.690443,1,-27.692365,27.692365,0.001583,158.756474
2026-04-06 09:00:00,2151.376484,1,-27.447786,27.447786,0.005914,2221.758072
2026-04-06 10:00:00,2152.431246,1,-27.427613,27.427613,0.000490,15.264403
2026-04-06 11:00:00,2150.092004,1,-27.472386,27.472386,-0.001087,75.083481


**RV / BV / Jump daily (last 10 days)**

In [6]:
rv_bv_view = compute_rv_bv_jumps(hf["r"], normalize_bv=True)
rv_bv_view.tail(10)[["rv_daily", "bv_raw", "bv_bns", "bv_daily", "jump_daily"]]


,rv_daily,bv_raw,bv_bns,bv_daily,jump_daily
hour,,,,,
2026-03-28,0.000349,0.000120,0.000189,0.000189,0.000160
2026-03-29,0.000444,0.000209,0.000328,0.000328,0.000117
2026-03-30,0.000661,0.000399,0.000627,0.000627,0.000033
2026-03-31,0.001361,0.000653,0.001026,0.001026,0.000335
2026-04-01,0.000846,0.000568,0.000893,0.000893,0.000000
2026-04-02,0.001377,0.000785,0.001234,0.001234,0.000143
2026-04-03,0.000104,0.000059,0.000093,0.000093,0.000011
2026-04-04,0.000067,0.000035,0.000054,0.000054,0.000013
2026-04-05,0.000493,0.000301,0.000473,0.000473,0.000021


**Daily IL decomposition (last 10 days)**

In [7]:
daily_df.tail(10)[["fees_daily", "il_daily", "bv_share", "jump_share", "il_diffusion", "il_jump", "gamma_pnl_daily"]]


,fees_daily,il_daily,bv_share,jump_share,il_diffusion,il_jump,gamma_pnl_daily
hour,,,,,,,
2026-03-28,44565.837138,21432.382755,0.540430,0.459570,11582.711826,9849.670930,23133.454383
2026-03-29,67514.179688,27075.536540,0.737723,0.262277,19974.235743,7101.300797,40438.643148
2026-03-30,110332.688755,40891.235269,0.949738,0.050262,38835.971042,2055.264227,69441.453486
2026-03-31,130559.664071,84662.445837,0.753948,0.246052,63831.106968,20831.338869,45897.218234
2026-04-01,165138.940637,53512.728109,1.000000,0.000000,53512.728109,0.000000,111626.212528
2026-04-02,112244.092442,85992.619964,0.896163,0.103837,77063.414678,8929.205286,26251.472478
2026-04-03,46549.342765,6440.577199,0.892906,0.107094,5750.833074,689.744126,40108.765566
2026-04-04,18230.108858,4175.061841,0.808465,0.191535,3375.391020,799.670820,14055.047017
2026-04-05,47141.227895,30784.944568,0.958208,0.041792,29498.371874,1286.572694,16356.283327


## §5 — Summary metrics

In [8]:
# σ* on TWO denominators (user spec uses TVL_active; we also compute V_FR for cross-check)
import math
mean_fees = summary['mean_fees']
sigma_star_active_daily  = math.sqrt(8.0 * mean_fees / tvl_active_avg) if mean_fees > 0 else float('nan')
sigma_star_active_annual = sigma_star_active_daily * math.sqrt(365.0)
sigma_star_fr_daily      = math.sqrt(8.0 * mean_fees / V_FR_avg) if mean_fees > 0 else float('nan')
sigma_star_fr_annual     = sigma_star_fr_daily * math.sqrt(365.0)

print("─" * 72)
print(f"  GAMMA-RISK SUMMARY  ({summary['n_days']} daily observations)")
print("─" * 72)
print(f"  mean fees / day:           ${summary['mean_fees']:>15,.2f}")
print(f"  mean IL / day:             ${summary['mean_IL']:>15,.2f}")
print(f"  std IL / day:              ${summary['std_IL']:>15,.2f}")
print()
print(f"  FGY = mean(F)/mean(IL):    {summary['FGY']:>15.4f}")
print(f"  GSR = mean(F)/std(IL):     {summary['GSR']:>15.4f}")
print()
print(f"  σ*_daily (TVL_active):     {sigma_star_active_daily:>15.6f}")
print(f"  σ*_annual (TVL_active):    {sigma_star_active_annual:>14.1%}   ← user spec (vs ${tvl_active_avg/1e6:.1f}M deployed)")
print(f"  σ*_daily (V_FR):           {sigma_star_fr_daily:>15.6f}")
print(f"  σ*_annual (V_FR):          {sigma_star_fr_annual:>14.1%}   ← break-even vs full-range value (${V_FR_avg/1e6:.1f}M)")
print()
print(f"  pct_profitable_days:       {summary['pct_profitable_days']:>14.1%}")
print(f"  total GammaPnL (sum):      ${summary['total_gamma_pnl']:>15,.0f}")
print(f"  worst 5 days sum:          ${summary['worst_5_days_sum']:>15,.0f}")
print(f"  worst 5 / total share:     {summary['tail_loss_contribution']:>15.2%}")
print(f"  skewness IL_daily:         {summary['skewness_il_daily']:>15.3f}")
print("─" * 72)

# Override summary dict with both σ*
summary_full = dict(summary)
summary_full["sigma_star_daily_TVL_active"]  = sigma_star_active_daily
summary_full["sigma_star_annual_TVL_active"] = sigma_star_active_annual
summary_full["sigma_star_daily_V_FR"]        = sigma_star_fr_daily
summary_full["sigma_star_annual_V_FR"]       = sigma_star_fr_annual
summary_full["V_FR_avg"]                     = V_FR_avg

# Display the dict
{k: (round(v, 6) if isinstance(v, float) else v) for k, v in summary_full.items() if not hasattr(v, '__iter__') or isinstance(v, str)}


────────────────────────────────────────────────────────────────────────
  GAMMA-RISK SUMMARY  (181 daily observations)
────────────────────────────────────────────────────────────────────────
  mean fees / day:           $      60,250.16
  mean IL / day:             $      88,844.87
  std IL / day:              $      82,539.99

  FGY = mean(F)/mean(IL):             0.6782
  GSR = mean(F)/std(IL):              0.7300

  σ*_daily (TVL_active):            0.120077
  σ*_annual (TVL_active):            229.4%   ← user spec (vs $33.4M deployed)
  σ*_daily (V_FR):                  0.028704
  σ*_annual (V_FR):                   54.8%   ← break-even vs full-range value ($585.0M)

  pct_profitable_days:                35.9%
  total GammaPnL (sum):      $     -5,175,642
  worst 5 days sum:          $     -1,497,356
  worst 5 / total share:              28.93%
  skewness IL_daily:                   1.962
────────────────────────────────────────────────────────────────────────


{'FGY': 0.67815,
 'GSR': 0.729951,
 'mean_IL': 88844.866453,
 'std_IL': 82539.988695,
 'mean_fees': 60250.158473,
 'sigma_star_daily': 0.120077,
 'sigma_star_annual': 2.294064,
 'tvl_active_avg': 33429510.69578,
 'pct_profitable_days': 0.359116,
 'tail_loss_contribution': 0.289308,
 'worst_5_days_sum': -1497355.654519,
 'skewness_il_daily': 1.961591,
 'total_gamma_pnl': -5175642.144229,
 'n_days': 181,
 'sigma_star_daily_TVL_active': 0.120077,
 'sigma_star_annual_TVL_active': 2.294064,
 'sigma_star_daily_V_FR': 0.028704,
 'sigma_star_annual_V_FR': 0.548393,
 'V_FR_avg': 585001209.297206}

## §6 — Bonus plots

**Plot 1 — IL_daily distribution + jump-day overlay**

In [9]:
il = daily_df["il_daily"].dropna()
jump_share = daily_df["jump_share"].reindex(il.index).fillna(0.0)
is_jump_day = jump_share > 0.30  # ≥30% of IL attributed to jumps

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=il[~is_jump_day].to_list(), name="normal days",
    nbinsx=30, marker_color="#3b82f6", opacity=0.75,
))
fig.add_trace(go.Histogram(
    x=il[is_jump_day].to_list(), name="jump days (jump_share > 30%)",
    nbinsx=30, marker_color="#ef4444", opacity=0.75,
))
fig.add_vline(x=float(il.mean()), line_dash="dash", line_color="white",
              annotation_text=f"mean=${il.mean():,.0f}")
fig.update_layout(
    barmode="overlay", height=380,
    title_text="Daily realized IL — distribution (jump days highlighted)",
    xaxis_title="IL_daily (USD)", yaxis_title="frequency",
)
fig.show()


**Plot 2 — Cumulative GammaPnL = Σ (Fees − IL)**

In [10]:
gpl = daily_df["gamma_pnl_daily"].dropna().cumsum()
fees_cum = daily_df["fees_daily"].fillna(0.0).cumsum()
il_cum = daily_df["il_daily"].fillna(0.0).cumsum()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=("Cumulative components", "Cumulative GammaPnL = Fees − IL"))
fig.add_trace(go.Scatter(x=fees_cum.index, y=fees_cum.values, name="Σ Fees", line=dict(color="#22c55e")),
              row=1, col=1)
fig.add_trace(go.Scatter(x=il_cum.index, y=(-il_cum).values, name="−Σ IL", line=dict(color="#ef4444")),
              row=1, col=1)
fig.add_trace(go.Scatter(x=gpl.index, y=gpl.values, name="Cumulative GammaPnL",
                          line=dict(color="#3b82f6", width=2)), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", row=2, col=1)
fig.update_layout(height=560, title_text=f"Cumulative GammaPnL — net = ${gpl.iloc[-1]:,.0f}")
fig.show()


**Plot 3 — Jump-vs-normal day contribution to total IL**

In [11]:
total_il = float(il.sum())
total_il_jump = float(daily_df["il_jump"].sum())
total_il_diff = float(daily_df["il_diffusion"].sum())
n_jump = int(is_jump_day.sum())
n_normal = int((~is_jump_day).sum())

print(f"Total IL              : ${total_il:>15,.0f}")
print(f"Total IL (diffusion)  : ${total_il_diff:>15,.0f}  ({total_il_diff/total_il:.1%})")
print(f"Total IL (jump)       : ${total_il_jump:>15,.0f}  ({total_il_jump/total_il:.1%})")
print(f"Jump days             : {n_jump} / {n_jump + n_normal}")

# Bar: per-day IL stacked diffusion + jump
fig = go.Figure()
fig.add_trace(go.Bar(x=daily_df.index, y=daily_df["il_diffusion"].fillna(0).values,
                      name="IL diffusion (BV)", marker_color="#3b82f6"))
fig.add_trace(go.Bar(x=daily_df.index, y=daily_df["il_jump"].fillna(0).values,
                      name="IL jump (RV − BV)", marker_color="#ef4444"))
fig.update_layout(barmode="stack", height=380,
    title_text="Daily IL — diffusion vs jump decomposition",
    xaxis_title="date", yaxis_title="IL_daily (USD)")
fig.show()


Total IL              : $     16,080,921
Total IL (diffusion)  : $     13,389,020  (83.3%)
Total IL (jump)       : $      2,691,901  (16.7%)
Jump days             : 32 / 181


## §6b — Range-width sensitivity

The framework above used a wide range data-fitted to the realized window's [P_min, P_max] expanded by ±20% margin. That's quasi-full-range — not a typical concentrated LP.

Here we compare **4 representative widths** centered on the window's geometric mean price (= √(P_min × P_max)) with **position fees corrected by the fraction of time the LP was in-range**:

```
position_fees_daily = pool_fees_daily × fraction_in_range_daily
```

Trade-off : **narrow** = higher gamma per dollar but less time active ; **wide** = robust but no concentration premium.


In [12]:
import math, numpy as np
P_geo_mean = math.sqrt(float(hf['price'].min()) * float(hf['price'].max()))

WIDTH_CASES = [
    ('narrow ±10%',   0.10),
    ('balanced ±30%', 0.30),
    ('wide ±60%',     0.60),
    ('fitted-wide',   None),  # data-fitted: [P_min×0.8, P_max×1.2]
]

rows = []
for label, width in WIDTH_CASES:
    if width is None:
        Pa_w = float(hf['price'].min()) * 0.80
        Pb_w = float(hf['price'].max()) * 1.20
    else:
        Pa_w = P_geo_mean * (1 - width)
        Pb_w = P_geo_mean * (1 + width)
    pos_w = LpPosition(L=L_human, Pa=Pa_w, Pb=Pb_w)

    # In-range mask (hourly) → daily fraction in-range
    in_range_hourly = ((hf['price'] >= Pa_w) & (hf['price'] <= Pb_w)).astype(float)
    fraction_in_range_daily = in_range_hourly.groupby(in_range_hourly.index.normalize()).mean()

    # Position fees correction
    fees_aligned = pd.Series(
        fees.values, index=pd.to_datetime(fees.index).normalize()
    ).reindex(fraction_in_range_daily.index).fillna(0.0)
    position_fees_daily = fees_aligned * fraction_in_range_daily

    out_w = run_gamma_risk_pipeline(
        prices=hf['price'], returns=hf['r'],
        fees_daily=position_fees_daily,
        position=pos_w, tvl_active_avg=tvl_active_avg,
    )
    s = out_w.summary
    log_width = math.log(Pb_w / Pa_w)
    rows.append({
        'width': label,
        'Pa': round(Pa_w, 0),
        'Pb': round(Pb_w, 0),
        'log(Pb/Pa)': round(log_width, 2),
        'days_in_range': int((fraction_in_range_daily > 0).sum()),
        'mean_in_range_pct': round(float(fraction_in_range_daily.mean()), 3),
        'mean_fees_d': round(s['mean_fees'], 0),
        'mean_IL_d': round(s['mean_IL'], 0),
        'std_IL_d': round(s['std_IL'], 0),
        'FGY': round(s['FGY'], 3) if s['FGY'] == s['FGY'] else None,
        'GSR': round(s['GSR'], 3) if s['GSR'] == s['GSR'] else None,
        'pct_profitable': round(s['pct_profitable_days'], 3),
        'total_PnL': round(s['total_gamma_pnl'], 0),
    })

comparison = pd.DataFrame(rows)
comparison


,width,Pa,Pb,log(Pb/Pa),days_in_range,mean_in_range_pct,mean_fees_d,mean_IL_d,std_IL_d,FGY,GSR,pct_profitable,total_PnL
0,narrow ±10%,2584.0,3159.0,0.20,70,0.347,7099.0,26379.0,55644.0,0.269,0.128,0.066,-3489567.0
1,balanced ±30%,2010.0,3733.0,0.62,147,0.728,38124.0,61769.0,77578.0,0.617,0.491,0.304,-4279831.0
2,wide ±60%,1149.0,4595.0,1.39,181,1.000,60250.0,88845.0,82540.0,0.678,0.730,0.359,-5175642.0
3,fitted-wide,1453.0,5449.0,1.32,181,1.000,60250.0,88845.0,82540.0,0.678,0.730,0.359,-5175642.0


**Reading guide**
- `log(Pb/Pa)` ≈ V3 standard width metric. 0.2 ≈ ±10%, 1.0 ≈ ±60%.
- `FGY > 1` → fees cover IL on average. `GSR > 1` → fees exceed 1σ of IL.
- `pct_profitable > 50%` and `total_PnL > 0` are necessary for an LP to be sustainable in this regime.


## §7 — Caveats & next steps

- BV uses the **BNS π/2 normalization** so `BV/RV → 1` under pure diffusion. The literal user spec without π/2 would mechanically attribute ~36% of IL to "jumps" even on calm days. The engine exposes both `bv_raw` and `bv_bns` for transparency.
- `Std[IL_daily]` is sample stddev (`ddof=1`). For thin windows (< 30 days) this is noisy; consider rolling 60-90d if extending.
- σ*_daily is computed from `mean(F_daily) / TVL_active_avg`. For a position-specific σ*, swap `TVL_active_avg` for the position's `2·L·√P_avg` — same number for a hypothetical full-pool LP.
- `worst_5 / total_pnl` is informative only when `total_pnl ≠ 0`; tail share is dimensionless ratio (can exceed 1 if total is small/negative).
- Synthetic position: `L = pool's average active L_raw / 10^((d0+d1)/2)`. For real positions, plug `L_human` from `mart_nfpm_position_pnl_current`.
- Pool was USDC/WETH 0.05% (not 0.3%). Concentration leverage on this tier is higher; the 0.3% pool would show smaller IL per dollar TVL but lower fee yield too.
